# Transaktions-Kategorisierung für Kontoauszüge

Dieses Notebook baut Schritt für Schritt ein ML-System, das Banktransaktionen
automatisch Kategorien zuordnet (z. B. *REWE* → `supermarkt`) — und dabei **neue
Kategorien zur Laufzeit dazulernen** kann.

## Architektur: 3 Stufen

| Stufe | Mechanismus | Deckt ab |
|---|---|---|
| 1 | **Merchant-Memory** — exakte/Prefix-Zuordnung bekannter Gegenparteien | wiederkehrende Transaktionen (~80 %) |
| 2 | **ML-Modell** — TF-IDF (Char-n-Gramme) + Logistische Regression | neue, aber ähnliche Händler |
| 3 | **Fallback** — Konfidenz unter Schwelle → `None` + Vorschlag | alles Unbekannte, User entscheidet |

Jede manuelle Kategorisierung fließt als Feedback zurück: Stufe 1 kennt den
Händler danach sofort, Stufe 2 wird neu trainiert.


In [1]:
from __future__ import annotations

import json
import re
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict, replace
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

## 1. Normalisierung deutscher Bank-Strings

Rohe Verwendungszwecke sind voller Rauschen, das dem Modell nichts bringt —
oder schlimmer: das es auswendig lernt statt zu generalisieren.

Typische Störquellen in deutschen SEPA-Exporten:

- **IBANs** (`DE89370400440532013000`) — eindeutig pro Konto, null Generalisierung
- **SEPA-Referenzfelder** (`EREF+`, `MREF+`, `SVWZ+`, `KREF+`, …) — technische Referenzen
- **Datumsangaben und lange Zahlen** — Kundennummern, Filialnummern, Automaten-IDs
- **Buchungstyp-Wörter** (`LASTSCHRIFT`, `KARTENZAHLUNG`, …) — stehen bei *jeder*
  Transaktion und tragen keine Kategorien-Information

Zusätzlich: Kleinschreibung und Umlaut-Transliteration (`ä→ae`), damit
`BÄCKEREI` und `Baeckerei` identisch aussehen.

In [2]:
_NOISE_PATTERNS = [
    re.compile(r"\b[A-Z]{2}\d{2}[A-Z0-9]{10,30}\b"),          # IBAN
    re.compile(r"\b(EREF|MREF|KREF|CRED|DEBT|SVWZ|ABWA|ABWE|BIC|OAMT|COAM)\+?\S*", re.I),
    re.compile(r"\b\d{2}[./]\d{2}[./]\d{2,4}\b"),              # Datumsangaben
    re.compile(r"\b(GA|EC)\s?NR\.?\s?\d+", re.I),              # Automaten-Nummern
    re.compile(r"\b(LASTSCHRIFT|GUTSCHRIFT|DAUERAUFTRAG|UEBERWEISUNG|ONLINE-UEBERWEISUNG|KARTENZAHLUNG|ENTGELTABRECHNUNG)\b", re.I),
    re.compile(r"\b\d{4,}\b"),                                 # lange Zahlen (Kundennr. etc.)
]

def normalize(text: str) -> str:
    """Kleinschreibung, Umlaut-Transliteration, Bank-Rauschen entfernen."""
    text = unicodedata.normalize("NFKC", text or "")
    text = text.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace("ß", "ss")
    text = text.lower()
    for pat in _NOISE_PATTERNS:
        text = pat.sub(" ", text)
    text = re.sub(r"[^a-z ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

In [3]:
# Was macht die Normalisierung konkret?
beispiele = [
    "REWE SAGT DANKE. 44123//KOELN/DE 2026-06-01T18:32 KARTENZAHLUNG",
    "Miete Juni SVWZ+RG-4711 IBAN DE89370400440532013000",
    "NETFLIX.COM MREF+X4Y7Z9 EREF+STRIPE-8834721",
    "Bäckerei Schmitz GA NR00042 EC 55221133",
]
for b in beispiele:
    print(f"  vorher : {b}")
    print(f"  nachher: {normalize(b)}\n")

  vorher : REWE SAGT DANKE. 44123//KOELN/DE 2026-06-01T18:32 KARTENZAHLUNG
  nachher: rewe sagt danke koeln de t

  vorher : Miete Juni SVWZ+RG-4711 IBAN DE89370400440532013000
  nachher: miete juni iban de

  vorher : NETFLIX.COM MREF+X4Y7Z9 EREF+STRIPE-8834721
  nachher: netflix com

  vorher : Bäckerei Schmitz GA NR00042 EC 55221133
  nachher: baeckerei schmitz ec



### Merchant-Key

Für das Memory (Stufe 1) brauchen wir einen **stabilen Schlüssel** pro Händler.
Problem: Derselbe Händler taucht in Varianten auf — `REWE Markt GmbH Filiale 4711`,
`REWE Markt GmbH Koeln`, … Wir nehmen deshalb nur die **ersten 3 Tokens** des
normalisierten Namens. Das reicht, um Händler zu unterscheiden, und kollabiert
Filialvarianten auf denselben Key.

In [4]:
def merchant_key(counterparty: str) -> str:
    """Stabiler Schlüssel für das Merchant-Memory (erste 3 Tokens)."""
    tokens = normalize(counterparty).split()
    return " ".join(tokens[:3])

for name in ["REWE Markt GmbH Filiale 4711", "REWE Markt GmbH Koeln", "ARAL Station 1234"]:
    print(f"  {name:32s} -> '{merchant_key(name)}'")

  REWE Markt GmbH Filiale 4711     -> 'rewe markt gmbh'
  REWE Markt GmbH Koeln            -> 'rewe markt gmbh'
  ARAL Station 1234                -> 'aral station'


## 2. Datentypen

Drei kleine Dataclasses:

- **`Transaction`** — was aus deinem Kontoauszug kommt. Das Vorzeichen des
  Betrags bestimmt den Typ (`income`/`expense`) — das nutzen wir später als
  harte Nebenbedingung: eine Ausgabe kann nie als „Gehalt" klassifiziert werden.
- **`Prediction`** — das Ergebnis. `category=None` heißt „User soll entscheiden",
  aber `suggestion` enthält trotzdem den besten Tipp, den deine UI vorauswählen kann.
- **`_Example`** — ein gespeichertes Trainingsbeispiel (intern).

In [5]:
@dataclass
class Transaction:
    counterparty: str          # Empfänger / Auftraggeber
    purpose: str               # Verwendungszweck
    amount: float              # negativ = Ausgabe, positiv = Einnahme

    @property
    def tx_type(self) -> str:
        return "income" if self.amount > 0 else "expense"

    def feature_text(self) -> str:
        return f"{normalize(self.counterparty)} {normalize(self.purpose)}"


@dataclass
class Prediction:
    category: str | None       # slug oder None (= manuell kategorisieren)
    confidence: float
    source: str                # "memory" | "model" | "none"
    suggestion: str | None = None  # bester Tipp, auch unter Schwelle (für UI-Vorauswahl)


@dataclass
class _Example:
    text: str                  # bereits normalisierter Feature-Text
    mkey: str                  # merchant key
    tx_type: str
    category: str

## 3. Das ML-Modell: TF-IDF mit Char-n-Grammen + Logistische Regression

### Warum Char-n-Gramme statt Wort-Tokens?

Verwendungszwecke enthalten zusammengeklebte Strings wie `NETFLIX.COMMREF` oder
`PAYPAL*SPOTIFY`. Eine Wort-Tokenisierung zerbricht daran. Char-n-Gramme
(`analyzer="char_wb"`, n = 2…5) zerlegen den Text in überlappende
Zeichenfolgen — `netf`, `etfl`, `tfli`, … — und erkennen so `netflix` auch dann,
wenn es in Müll eingebettet ist. `char_wb` respektiert dabei Wortgrenzen
(n-Gramme laufen nicht über Leerzeichen hinweg), was das Vokabular sauber hält.

### Warum Logistische Regression?

- liefert **kalibrierte Wahrscheinlichkeiten** (`predict_proba`) — die brauchen
  wir für die Konfidenz-Schwelle in Stufe 3
- trainiert auf tausenden Beispielen in Millisekunden → Voll-Retrain ist billig
- `class_weight="balanced"` gewichtet seltene Klassen hoch. Das ist der
  entscheidende Punkt für **neue Kategorien**: Eine frische Kategorie mit 2
  Beispielen darf gegen `supermarkt` mit 500 Beispielen nicht untergehen.

### Die Typ-Maske

Nach `predict_proba` setzen wir die Wahrscheinlichkeit aller Kategorien, deren
Typ nicht zum Vorzeichen passt, auf 0 und **renormalisieren**. Die Konfidenz ist
danach also „Wahrscheinlichkeit *innerhalb der plausiblen Kategorien*" — eine
Ausgabe konkurriert nur noch mit Ausgabe-Kategorien.

## 4. Der Classifier

Die zentrale Klasse. Wichtige Design-Entscheidungen:

**Lazy Retraining** (`_dirty`-Flag): `add_feedback()` markiert das Modell nur
als veraltet. Erst der nächste `predict()`-Aufruf trainiert neu. So kannst du
50 Transaktionen am Stück kategorisieren, ohne 50-mal zu trainieren.

**Prefix-Matching im Memory**: `ARAL` soll das gespeicherte `aral station`
matchen — aber `Deutsche Bank` darf *nicht* `Deutsche Bahn` matchen. Die Regel:
Match nur, wenn die Token-Liste des einen ein **Präfix** der anderen ist.
Bei `["deutsche","bank"]` vs. `["deutsche","bahn"]` weichen die zweiten Tokens
voneinander ab → kein Match. Bei `["aral"]` vs. `["aral","station"]` ist die
kürzere Liste ein Präfix der längeren → Match.

**Neue Kategorien**: `add_feedback()` mit unbekanntem Slug registriert die
Kategorie implizit (`setdefault`). Kein separater Registrierungsschritt nötig —
`add_category()` gibt es trotzdem, falls deine App Kategorien anlegt, bevor
Beispiele existieren.

**Mindestdaten**: Unter 4 Beispielen oder mit nur 1 Klasse trainieren wir gar
nicht erst — dann arbeitet nur das Memory. Verhindert Unsinn in der Kaltstart-Phase.

In [6]:
class TransactionClassifier:
    def __init__(
        self,
        category_types: dict[str, str],   # slug -> "income" | "expense"
        confidence_threshold: float = 0.55,
        memory_min_count: int = 1,        # ab wie vielen Bestätigungen Memory greift
    ):
        self.category_types = dict(category_types)
        self.threshold = confidence_threshold
        self.memory_min_count = memory_min_count

        self._examples: list[_Example] = []
        self._memory: dict[str, Counter] = defaultdict(Counter)  # mkey -> Counter(category)
        self._pipeline: Pipeline | None = None
        self._dirty = False               # lazy retrain

    # -- Kategorien-Verwaltung ------------------------------------------

    def add_category(self, slug: str, tx_type: str) -> None:
        """Neue Kategorie registrieren. Gelernt wird sie, sobald das erste
        Feedback-Beispiel dafür eintrifft."""
        self.category_types[slug] = tx_type

    def remove_category(self, slug: str, reassign_to: str | None = None) -> None:
        """Kategorie löschen; Trainingsbeispiele umhängen oder verwerfen."""
        self.category_types.pop(slug, None)
        if reassign_to:
            for ex in self._examples:
                if ex.category == slug:
                    ex.category = reassign_to
        else:
            self._examples = [ex for ex in self._examples if ex.category != slug]
        self._rebuild_memory()
        self._dirty = True

    # -- Lernen -----------------------------------------------------------

    def add_feedback(self, tx: Transaction, category: str) -> None:
        """User hat eine Transaktion (neu) kategorisiert -> lernen.
        Unbekannte Kategorien werden implizit registriert."""
        self.category_types.setdefault(category, tx.tx_type)
        ex = _Example(
            text=tx.feature_text(),
            mkey=merchant_key(tx.counterparty),
            tx_type=tx.tx_type,
            category=category,
        )
        self._examples.append(ex)
        if ex.mkey:
            self._memory[ex.mkey][category] += 1
        self._dirty = True

    def _rebuild_memory(self) -> None:
        self._memory = defaultdict(Counter)
        for ex in self._examples:
            if ex.mkey:
                self._memory[ex.mkey][ex.category] += 1

    def _retrain(self) -> None:
        self._dirty = False
        labels = {ex.category for ex in self._examples}
        if len(self._examples) < 4 or len(labels) < 2:
            self._pipeline = None         # zu wenig Daten -> nur Memory nutzen
            return
        self._pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(
                analyzer="char_wb",       # Char-n-Gramme: robust bei Merchant-Strings
                ngram_range=(2, 5),
                min_df=1,
                sublinear_tf=True,
            )),
            ("clf", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",  # neue/seltene Kategorien nicht untergehen lassen
                C=5.0,
            )),
        ])
        X = [ex.text for ex in self._examples]
        y = [ex.category for ex in self._examples]
        self._pipeline.fit(X, y)

    # -- Vorhersage ---------------------------------------------------------

    def predict(self, tx: Transaction) -> Prediction:
        if self._dirty:
            self._retrain()

        # Stufe 1: Merchant-Memory (Prefix-Match in beide Richtungen)
        q = merchant_key(tx.counterparty).split()
        best_match, best_len = None, 0
        if q:
            for key, counts in self._memory.items():
                k = key.split()
                n_common = min(len(q), len(k))
                if q[:n_common] == k[:n_common] and n_common > best_len:
                    best_match, best_len = counts, n_common
        if best_match is not None:
            best, n = best_match.most_common(1)[0]
            total = sum(best_match.values())
            # nur wenn Kategorie-Typ zum Vorzeichen passt
            if n >= self.memory_min_count and self.category_types.get(best) == tx.tx_type:
                return Prediction(best, n / total, "memory")

        # Stufe 2: ML-Modell (mit Typ-Maske)
        if self._pipeline is not None:
            proba = self._pipeline.predict_proba([tx.feature_text()])[0]
            classes = self._pipeline.named_steps["clf"].classes_
            mask = np.array([
                self.category_types.get(c) == tx.tx_type for c in classes
            ])
            if mask.any():
                proba = np.where(mask, proba, 0.0)
                s = proba.sum()
                if s > 0:
                    proba = proba / s     # renormalisieren innerhalb des Typs
                    idx = int(np.argmax(proba))
                    conf = float(proba[idx])
                    if conf >= self.threshold:
                        return Prediction(str(classes[idx]), conf, "model")
                    # Stufe 3: unsicher -> User fragen, aber Vorschlag mitgeben
                    return Prediction(None, conf, "none", suggestion=str(classes[idx]))

        return Prediction(None, 0.0, "none")

    def predict_batch(self, txs: list[Transaction]) -> list[Prediction]:
        return [self.predict(tx) for tx in txs]

    # -- Persistenz -----------------------------------------------------------

    def save(self, path: str | Path) -> None:
        data = {
            "category_types": self.category_types,
            "threshold": self.threshold,
            "examples": [asdict(ex) for ex in self._examples],
        }
        Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=1))

    @classmethod
    def load(cls, path: str | Path) -> "TransactionClassifier":
        data = json.loads(Path(path).read_text())
        clf = cls(data["category_types"], confidence_threshold=data["threshold"])
        clf._examples = [_Example(**ex) for ex in data["examples"]]
        clf._rebuild_memory()
        clf._dirty = True
        return clf

## 5. Demo: Kaltstart mit Seed-Daten

Wir simulieren, was in deiner App beim ersten CSV-Import passiert: Der User
kategorisiert ein paar Transaktionen von Hand, jede Korrektur wird zu Feedback.

Das `category_types`-Dict leitest du direkt aus deinen `DEFAULT_CATEGORIES` ab:

```python
category_types = {slug: typ for slug, _, _, typ, _ in DEFAULT_CATEGORIES}
```

Hier eine reduzierte Version für die Demo:

In [7]:
category_types = {
    "supermarkt": "expense", "baeckerei": "expense", "tanken": "expense",
    "opnv": "expense", "streaming": "expense", "restaurant": "expense",
    "miete": "expense", "gehalt": "income", "zinsen": "income",
}
clf = TransactionClassifier(category_types)

seed = [
    (Transaction("REWE Markt GmbH", "REWE SAGT DANKE. 44123//KOELN/DE", -34.50), "supermarkt"),
    (Transaction("EDEKA Zentrale", "EDEKA DANKT 08154711", -22.10), "supermarkt"),
    (Transaction("ALDI SUED", "KARTENZAHLUNG 01.06.2026", -18.75), "supermarkt"),
    (Transaction("Baeckerei Schmitz", "EC 55221133 KAUFUMSATZ", -4.20), "baeckerei"),
    (Transaction("ARAL Station 1234", "ARAL TANKSTELLE DANKE", -65.00), "tanken"),
    (Transaction("Shell Deutschland", "SHELL 0815 KOELN", -58.30), "tanken"),
    (Transaction("DB Vertrieb GmbH", "Deutschlandticket Abo Juni", -58.00), "opnv"),
    (Transaction("Netflix International", "NETFLIX.COM MREF+ABC123", -13.99), "streaming"),
    (Transaction("DLR e.V.", "GEHALT 06/2026 PERSONALNR 4711", 2450.00), "gehalt"),
    (Transaction("Trattoria Milano", "KARTENZAHLUNG DANKE", -42.80), "restaurant"),
]
for tx, cat in seed:
    clf.add_feedback(tx, cat)

print(f"{len(clf._examples)} Trainingsbeispiele, {len(set(e.category for e in clf._examples))} Kategorien")

10 Trainingsbeispiele, 7 Kategorien


### Vorhersagen auf neuen Transaktionen

Vier Testfälle, die die drei Stufen zeigen:

1. **REWE Filiale 99** — bekannter Händler, neue Filiale → Memory (Prefix-Match)
2. **PENNY** — nie gesehen, aber der Verwendungszweck ähnelt anderen Supermärkten
3. **TotalEnergies** — nie gesehen, „Tankstelle" im Text ähnelt ARAL/Shell
4. **Unbekannter Empfänger** — nichts passt → `None`, User muss ran

Erwartung bei nur 10 Trainingsbeispielen: Das Modell ist **zu Recht unsicher**
(Konfidenz unter der Schwelle von 0.55) und liefert die Kategorien als
`suggestion` statt als feste Zuordnung. Das ist gewolltes, ehrliches Verhalten —
mit wachsenden Daten steigen die Konfidenzen und immer mehr läuft automatisch.

In [8]:
tests = [
    Transaction("REWE Markt GmbH Filiale 99", "REWE DANKT", -12.30),
    Transaction("PENNY Markt", "PENNY SAGT DANKE 4711", -9.80),
    Transaction("TotalEnergies Tankstelle", "TOTAL STATION KOELN", -70.00),
    Transaction("Unbekannter Empfaenger XY", "RECHNUNG 12345", -99.00),
]
for tx in tests:
    p = clf.predict(tx)
    extra = f", Vorschlag: {p.suggestion}" if p.suggestion else ""
    print(f"  {tx.counterparty:32s} -> {str(p.category):12s} ({p.confidence:.2f}, {p.source}{extra})")

  REWE Markt GmbH Filiale 99       -> supermarkt   (1.00, memory)
  PENNY Markt                      -> None         (0.40, none, Vorschlag: supermarkt)
  TotalEnergies Tankstelle         -> None         (0.52, none, Vorschlag: tanken)
  Unbekannter Empfaenger XY        -> None         (0.28, none, Vorschlag: supermarkt)


### Die Typ-Maske in Aktion

Eine *Einnahme* von der DLR wird nie als `supermarkt` klassifiziert, selbst
wenn der Text mehrdeutig wäre — Income-Beträge konkurrieren nur mit
Income-Kategorien:

In [9]:
p = clf.predict(Transaction("DLR e.V.", "GEHALT 07/2026 PERSONALNR 4711", 2450.00))
print(f"  DLR Gehalt (Einnahme) -> {p.category} ({p.confidence:.2f}, {p.source})")

  DLR Gehalt (Einnahme) -> gehalt (1.00, memory)


## 6. Der Kern-Usecase: neue Kategorie zur Laufzeit

Jetzt der Moment, für den die Architektur gebaut ist. Angenommen, du legst in
der App die Kategorie **`haustier`** an und kategorisierst zwei Transaktionen.
Kein Sonderfall, keine Migration — einfach Feedback geben:

In [10]:
clf.add_feedback(Transaction("Fressnapf GmbH", "FRESSNAPF FILIALE 42", -25.00), "haustier")
clf.add_feedback(Transaction("Tierarztpraxis Dr. Meier", "RECHNUNG BEHANDLUNG", -80.00), "haustier")

print("Registrierte Kategorien:", sorted(clf.category_types))
print()

# Bekannter Händler -> Memory greift sofort:
p = clf.predict(Transaction("Fressnapf GmbH", "KAUFUMSATZ", -19.99))
print(f"  Fressnapf GmbH        -> {p.category} ({p.confidence:.2f}, {p.source})")

# Das Modell kennt die Klasse jetzt auch (dank class_weight='balanced'
# trotz nur 2 Beispielen konkurrenzfähig):
p = clf.predict(Transaction("Zooplus SE", "TIERFUTTER BESTELLUNG", -31.50))
extra = f", Vorschlag: {p.suggestion}" if p.suggestion else ""
print(f"  Zooplus (neu)         -> {str(p.category)} ({p.confidence:.2f}, {p.source}{extra})")

Registrierte Kategorien: ['baeckerei', 'gehalt', 'haustier', 'miete', 'opnv', 'restaurant', 'streaming', 'supermarkt', 'tanken', 'zinsen']

  Fressnapf GmbH        -> haustier (1.00, memory)
  Zooplus (neu)         -> None (0.28, none, Vorschlag: haustier)


## 7. Persistenz

Gespeichert werden **nur die Trainingsbeispiele als JSON** — nicht das
gepickelte sklearn-Modell. Zwei Gründe:

1. **Keine Versionsprobleme**: Gepickelte sklearn-Objekte brechen gern bei
   Bibliotheks-Updates. Rohdaten + Retrain beim Laden ist immun dagegen.
2. **Passt in deine App-Datenbank**: Die Beispiele sind eine simple Tabelle
   (`text`, `mkey`, `tx_type`, `category`) — direkt abbildbar in SQLite/sql.js.

In [11]:
clf.save("/tmp/clf_state.json")
clf2 = TransactionClassifier.load("/tmp/clf_state.json")

p = clf2.predict(Transaction("ARAL", "TANKUNG", -50.0))
print(f"  Nach Reload: ARAL -> {p.category} ({p.confidence:.2f}, {p.source})")
print(f"  Dateigroesse: {Path('/tmp/clf_state.json').stat().st_size} Bytes")

  Nach Reload: ARAL -> tanken (1.00, memory)
  Dateigroesse: 1933 Bytes


## 8. Wie sich das System mit mehr Daten verhält

Ein kleines Experiment: Wir duplizieren die Seed-Daten mit leichten Variationen
(simuliert mehrere Monate Kontoauszug) und schauen, wie die Modell-Konfidenz
für den PENNY-Fall steigt:

In [12]:
for n_monate in [1, 3, 6, 12]:
    c = TransactionClassifier(category_types)
    for monat in range(n_monate):
        for tx, cat in seed:
            t = replace(tx, purpose=f"{tx.purpose} {monat:02d}/2026")
            c.add_feedback(t, cat)
    p = c.predict(Transaction("PENNY Markt", "PENNY SAGT DANKE 4711", -9.80))
    label = p.category or f"None (Vorschlag: {p.suggestion})"
    print(f"  {n_monate:2d} Monat(e) Daten -> {label:30s} Konfidenz {p.confidence:.2f}")

   1 Monat(e) Daten -> None (Vorschlag: supermarkt)   Konfidenz 0.40
   3 Monat(e) Daten -> supermarkt                     Konfidenz 0.57
   6 Monat(e) Daten -> supermarkt                     Konfidenz 0.66
  12 Monat(e) Daten -> supermarkt                     Konfidenz 0.74


Man sieht das gewünschte Verhalten: Am Anfang ist das System vorsichtig und
macht nur Vorschläge; mit mehr bestätigten Daten überschreitet die Konfidenz
die Schwelle und die Kategorisierung läuft automatisch. Die Schwelle
(`confidence_threshold`) ist dein Regler zwischen „lieber fragen" und
„lieber automatisch" — ein guter UX-Default ist, sie in den App-Einstellungen
justierbar zu machen.

## 9. Integration in deine App — Hinweise

**Feedback-Loop richtig anschließen.** Jede der drei UI-Aktionen ist Feedback:
(a) User bestätigt einen Vorschlag, (b) User korrigiert eine automatische
Zuordnung, (c) User kategorisiert eine `None`-Transaktion. Alle drei →
`add_feedback()`. Besonders (b) ist wichtig: Korrekturen sind die wertvollsten
Trainingssignale.

**Unterkategorien.** Das Modell lernt auf den Blatt-Slugs (`supermarkt`, nicht
`lebensmittel`). Die Eltern-Zuordnung ist reine Anzeige-Logik deiner App —
das Modell muss davon nichts wissen.

**Wenn die App nicht in Python läuft** (dein Entwurf zielt ja auf
macOS + GrapheneOS, offline-first): Die Architektur ist 1:1 übertragbar.
Das Merchant-Memory ist ein Dictionary, TF-IDF + Logistische Regression gibt
es auch als reine JS/TS-Implementierungen — oder du trainierst in Python und
exportierst nur die Gewichte (eine Matrix + Bias-Vektor, die Inferenz ist ein
Matrix-Vektor-Produkt). Erfahrungsgemäß deckt das Memory allein schon den
Großteil ab, weil Transaktionen so repetitiv sind.

**Nächster sinnvoller Schritt:** den Classifier auf einen echten CSV-Export
deiner Bank loslassen. Die `_NOISE_PATTERNS` sind auf typische SEPA-Formate
ausgelegt, aber jede Bank hat Eigenheiten im Verwendungszweck-Format.